In [ ]:
import pandas as pd 
import numpy as np 

#Importamos los archivos csv 1
#Datos macroeconomicos (PBI , Inflacion, Deuda , Resultados comercial y fiscal)
cs_macro = pd.read_csv(r"C:\Users\HP\Desktop\Universidad\Tec CsDatos\Analisis y Visualizacion de Datos\ARCHIVOS\AP2\datos_macro.csv")

#Tipos de cambio de Brasil y de Argentina (Precio del dolar reales y pesos argentinos)
tc_brasil = pd.read_csv(r"C:\Users\HP\Desktop\Universidad\Tec CsDatos\Analisis y Visualizacion de Datos\ARCHIVOS\AP2\tc_brasil.csv")
tc_argentina = pd.read_csv(r"C:\Users\HP\Desktop\Universidad\Tec CsDatos\Analisis y Visualizacion de Datos\ARCHIVOS\AP2\tc_argentina.csv")

#Chequear las primeras filas 2
#cs_macro.head()
#cs_macro.tail() #Ultimas filas
#tc_brasil.head()
#tc_argentina.head()

#Usamos shape para conocer la cantidad de filas y columnas 3
#print({"Filas y columnas de cs_macro" : cs_macro.shape})

#Chequear tipos de objetos (variables) 4
#cs_macro.dtypes

## FILTROS DE COLUMNAS Y FILAS , INTEGRACION DE DATOS Y LIMPIEZA DE DATOS ##

    # Importamos de vuelta el arch pero sin ultimas 3 filas 
macro_csv = pd.read_csv(r"C:\Users\HP\Desktop\Universidad\Tec CsDatos\Analisis y Visualizacion de Datos\ARCHIVOS\AP2\datos_macro.csv", skipfooter=3, engine='python')

    # Filtramos las observaciones/filas que correspondan los paises y variables
macro_csv_f = macro_csv[(macro_csv['Country'].isin(['Argentina','Chile','Brazil'])) & 
                        (macro_csv['Subject Descriptor'] == "Gross domestic product, constant prices") &
                        (macro_csv["Units"] == "Percent change" )]

    # Eliminamos las columnas que no necesitamos
macro_csv_f = macro_csv_f.drop(columns= ['WEO Country Code' , 'ISO' , 'WEO Subject Code',
                                        'Subject Descriptor' , 'Subject Notes' , 'Units' ,'Scale','Country/Series-specific Notes' , 'Estimates Start After'])

    #Transponemos el dataframe para que sea mas facil de procesar
#Convertimos los años en filas y definiendo la variable de valores creciente
macro_tr = pd.melt(
                    macro_csv_f,
                   id_vars=['Country'] , # columnas que se mantienen fijas
                   var_name = 'year' , #nombre de la nueva columna que contendrá los nombres de las columnas originales, 
                   value_name = 'crecimiento_pbi' , #nombre de la nueva columna que contendrá los valores
                   )

    #Renombramos la variable country por paismacro_tr.
macro_tr = macro_tr.rename(columns = {'Country' : 'pais'})

    #Ordenamos las filas pais por año
macro_tr = macro_tr.sort_values(by = ['pais' , 'year'])

    ## CHEQUO DE TIPOS DE VARIABLES Y CMABIAMOS ## 
macro_tr.dtypes
tc_brasil.dtypes

    #Convertimos las variables
macro_tr = macro_tr.astype({'pais' : str , 'year' : int , 'crecimiento_pbi' : float})

    #Filtramos las observaciones anteriores a 2020
macro_tr = macro_tr[macro_tr['year'] <= 2020]

    ## CALCULAMOS LAS PRINCIPALES ESTADISTICA DESCRIPTIVAS ## 
macro_tr['crecimiento_pbi'].describe()

    #Concatenamos las tablas con el tipo de cambio de Brasil y Argentina 
tc = pd.concat([tc_brasil , tc_argentina])
    #Combinamos las dos tablas
macro_tc = pd.merge(macro_tr, tc , left_on = ['pais', 'year'] , right_on = ['pais', 'year'] , how = 'outer')
macro_tc

    ## AGREGACIONES ##
    #Calculamos la media tasa de crecimiento por pais 
macro_ag = macro_tr.groupby(['pais'], as_index = False).agg({'crecimiento_pbi' : 'mean'})

    ### CHEQUEOS DE CALIDAD Y LIMPIEZA ###

    ##MISSINGS##
    #Contamos la cantiad de valores faltantes
macro_tc["tipo_de_cambio"].isnull().sum()

     # Eliminar las filas en los que aparezca por lo menos un missing
tc_ar = macro_tc.dropna(how = 'any')

    # Eliminar las filas en las que haya missing en tipo_de_cambio
tc_ar = macro_tc.dropna(subset = ['tipo_de_cambio'])

        ## INCONSISTENCIAS ##
    # Analizamos los valors unicos de la variable pais de la tabla macro_tr
macro_tr['pais'].unique()

    # Reemplazo 
    
tc_brasil['pais'] = np.where(tc_brasil['pais'] == 'Brazil' , 'Brasil', tc_brasil['pais'])
tc_brasil

        ### UOUTLIERS ###
    # Identificarlos 
tc_brasil['tc_estandarizado']  = (tc_brasil['tipo_de_cambio'] - np.mean(tc_brasil['tipo_de_cambio'])) / np.std(tc_brasil['tipo_de_cambio'])
tc_brasil

    # Conservamos solo los años desde 1995
tc_bra1 = tc_brasil[tc_brasil['year'] >= 1995]
tc_bra1

    #Percentil 99 de los valores tipo de cambio
p99 = np.percentile(tc_brasil['tipo_de_cambio'] , 99)
p99

    # Vemos las observaciones por encima del percentil 99 
tc_brasil[tc_brasil['tipo_de_cambio'] >= p99]

        ### CORRECCION E IMPUTACION ###
    

,pais,year,tipo_de_cambio,tc_estandarizado
0,Brasil,1994,537.595539,5.195833
